#1. Carga y preparación de datos (ETL)

In [ ]:
# 1.1 Librerías y config (ruta desde GASES_DATA_BASE_DIR / Secret Manager en cloud)
import sys
from pathlib import Path

_cwd = Path(".").resolve()
_project_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from config import PROCESSED_DATASET_PATH

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy.stats import gaussian_kde

# 1.2 Cargar archivo (data/processed vía config: local o descargado desde GCS)
path_procesamiento_datos = PROCESSED_DATASET_PATH
if str(path_procesamiento_datos).endswith(".csv"):
    df = pd.read_csv(path_procesamiento_datos)
else:
    df = pd.read_excel(path_procesamiento_datos)

# 1.3 Limpieza de columna numérica
df["Leyendo"] = df["Leyendo"].astype(str).str.replace(",", ".").astype(float)

# 1.4 Convertir fechas
df["Tiempo de Inicio"] = pd.to_datetime(df["Tiempo de Inicio"])

# 1.5 Ordenar cronológicamente
df = df.sort_values("Tiempo de Inicio")

df.head()

,Descripción,Sensor,Número de Periodo,Gas,Leyendo,Unidad,STEL Máx,TWA máximo,Tiempo de Inicio,Tiempo de finalización,Sitio,Factor de Respuesta,Factor,Alarma,Temperatura
20,O2,22101YV009#S0020,Periodo 1,Oxígeno,21.0,%vol,NaN,NaN,2024-09-06 12:53:44,2024-09-06 12:54:44,NaN,NaN,NaN,NaN,29
1,NO2,22083AF012#S0004,Periodo 1,Dióxido de nitrógeno,0.0,ppm,NaN,NaN,2024-09-06 12:53:44,2024-09-06 12:54:34,NaN,NaN,NaN,NaN,29
17,LEL(CH4),22101J7044#S0021,Periodo 1,Gas combustible,0.0,%vol,NaN,NaN,2024-09-06 12:53:44,2024-09-06 12:54:44,NaN,NaN,Metano,NaN,29
14,H2S,2209366013#S0002,Periodo 1,Sulfuro de hidrógeno,0.0,ppm,NaN,NaN,2024-09-06 12:53:44,2024-09-06 12:54:44,NaN,NaN,NaN,NaN,29
11,CO,2209366013#S0001,Periodo 1,Monóxido de carbono,0.0,ppm,NaN,NaN,2024-09-06 12:53:44,2024-09-06 12:54:44,NaN,NaN,NaN,NaN,29


#2. EDA – Análisis estadístico descriptivo

In [ ]:
df["Leyendo"].describe()

,Leyendo
count,2001.000000
mean,6.033868
std,8.151505
min,-0.600000
25%,0.120000
50%,1.010000
75%,16.100000
max,21.900000


2.1 Estadísticos generales

In [ ]:
df["Leyendo"].describe()

,Leyendo
count,2001.000000
mean,6.033868
std,8.151505
min,-0.600000
25%,0.120000
50%,1.010000
75%,16.100000
max,21.900000


2.2 Estadísticos por gas

In [ ]:
df.groupby("Gas")["Leyendo"].describe()

,count,mean,std,min,25%,50%,75%,max
Gas,,,,,,,,
Dióxido de carbono,853.0,1.425041,1.329741,-0.17,0.59,1.00,2.270,5.0
Dióxido de nitrógeno,52.0,0.026923,0.211545,-0.60,0.00,0.00,0.125,0.4
Gas combustible,246.0,0.090691,0.054044,0.00,0.05,0.11,0.130,0.3
Monóxido de carbono,26.0,0.961538,1.455494,0.00,0.00,0.00,2.000,5.0
Oxígeno,577.0,18.605026,2.142953,0.00,17.30,19.30,19.900,21.9
Sulfuro de hidrógeno,247.0,0.301215,0.351822,-0.60,0.00,0.30,0.500,1.4


2.3 Conteo de registros por gas

In [ ]:
df["Gas"].value_counts()

,count
Gas,
Dióxido de carbono,853
Oxígeno,577
Sulfuro de hidrógeno,247
Gas combustible,246
Dióxido de nitrógeno,52
Monóxido de carbono,26


#3️. EDA – Análisis de distribuciones

3.1 Histogramas por gas

In [ ]:
gases = df["Gas"].unique()

for gas in gases:
    subset = df[df["Gas"] == gas]["Leyendo"].dropna()

    if len(subset) > 1:
        kde = gaussian_kde(subset)
        x_range = np.linspace(subset.min(), subset.max(), 200)

        fig = go.Figure()

        fig.add_trace(go.Histogram(
            x=subset,
            nbinsx=30,
            histnorm="density",
            name="Frecuencia",
            marker_color="steelblue",
            opacity=0.6
        ))

        fig.add_trace(go.Scatter(
            x=x_range,
            y=kde(x_range),
            mode="lines",
            name="KDE",
            line=dict(color="crimson", width=2)
        ))

        fig.update_layout(
            title=f"Distribución de concentración — {gas}",
            xaxis_title="Concentración",
            yaxis_title="Densidad",
            template="plotly_white",
            hovermode="x unified"
        )

        fig.show()

3.2 Boxplot interactivo

In [ ]:
fig_box = px.box(
    df,
    x="Gas",
    y="Leyendo",
    color="Gas",
    template="plotly_white",
    title="Detección de Outliers por Gas"
)

fig_box.update_layout(showlegend=False)
fig_box.show()

#4. Análisis Temporal

4.1 Serie temporal por gas

In [ ]:
for gas in gases:
    subset = df[df["Gas"] == gas]

    fig = px.line(
        subset,
        x="Tiempo de Inicio",
        y="Leyendo",
        title=f"Serie Temporal — {gas}",
        template="plotly_white"
    )

    fig.show()

4.2 Tendencia con media móvil

In [ ]:
df["Media_Movil"] = df["Leyendo"].rolling(window=20).mean()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["Tiempo de Inicio"],
    y=df["Leyendo"],
    mode="lines",
    name="Valor original",
    line=dict(color="lightgray")
))

fig.add_trace(go.Scatter(
    x=df["Tiempo de Inicio"],
    y=df["Media_Movil"],
    mode="lines",
    name="Media móvil",
    line=dict(color="red", width=2)
))

fig.update_layout(
    title="Tendencia con Media Móvil",
    template="plotly_white"
)

fig.show()

4.3 Detección de cambios abruptos

In [ ]:
df["Cambio"] = df["Leyendo"].diff().abs()

threshold = df["Cambio"].mean() + 2 * df["Cambio"].std()
df["Cambio_Brusco"] = df["Cambio"] > threshold

df[df["Cambio_Brusco"] == True].head()

,Descripción,Sensor,Número de Periodo,Gas,Leyendo,Unidad,STEL Máx,TWA máximo,Tiempo de Inicio,Tiempo de finalización,Sitio,Factor de Respuesta,Factor,Alarma,Temperatura,Media_Movil,Cambio,Cambio_Brusco


#5. EDA – Análisis multivariable

5.1 Pivotear datos por gas

In [ ]:
df_pivot = df.pivot_table(
    index="Tiempo de Inicio",
    columns="Gas",
    values="Leyendo"
)

5.2 Matriz de correlación interactiva

In [ ]:
fig_corr = px.imshow(
    df_pivot.corr(),
    text_auto=True,
    color_continuous_scale="RdBu_r",
    title="Matriz de Correlación entre Gases"
)

fig_corr.show()

#6. EDA – Patrones previos a eventos

6.1 Definir evento (Percentil 95)

In [ ]:
limite = df["Leyendo"].quantile(0.95)
eventos = df[df["Leyendo"] > limite]

eventos.head()

,Descripción,Sensor,Número de Periodo,Gas,Leyendo,Unidad,STEL Máx,TWA máximo,Tiempo de Inicio,Tiempo de finalización,Sitio,Factor de Respuesta,Factor,Alarma,Temperatura,Media_Movil,Cambio,Cambio_Brusco
20,O2,22101YV009#S0020,Periodo 1,Oxígeno,21.000000,%vol,NaN,NaN,2024-09-06 12:53:44,2024-09-06 12:54:44,NaN,NaN,NaN,NaN,29,NaN,NaN,False
24,O2,22101YV009#S0020,Periodo 2,Oxígeno,21.900000,%vol,NaN,NaN,2024-09-06 14:40:40,2024-09-06 14:41:20,NaN,NaN,NaN,NaN,26,NaN,21.800000,False
23,O2,22101YV009#S0020,Periodo 2,Oxígeno,21.799999,%vol,NaN,NaN,2024-09-06 14:41:20,2024-09-06 14:41:30,NaN,NaN,NaN,NaN,26,NaN,21.799999,False
22,O2,22101YV009#S0020,Periodo 2,Oxígeno,21.900000,%vol,NaN,NaN,2024-09-06 14:41:30,2024-09-06 14:41:40,NaN,NaN,NaN,NaN,26,NaN,0.100000,False
21,O2,22101YV009#S0020,Periodo 2,Oxígeno,21.799999,%vol,NaN,NaN,2024-09-06 14:41:40,2024-09-06 14:43:40,NaN,NaN,NaN,NaN,26,NaN,0.100000,False


6.2 Analizar ventana previa de 30 minutos

In [ ]:
ventanas = []

for tiempo in eventos["Tiempo de Inicio"]:
    ventana = df[
        (df["Tiempo de Inicio"] >= tiempo - pd.Timedelta(minutes=30)) &
        (df["Tiempo de Inicio"] <= tiempo)
    ]
    ventanas.append(ventana)

df_ventanas = pd.concat(ventanas)

df_ventanas.head()

,Descripción,Sensor,Número de Periodo,Gas,Leyendo,Unidad,STEL Máx,TWA máximo,Tiempo de Inicio,Tiempo de finalización,Sitio,Factor de Respuesta,Factor,Alarma,Temperatura,Media_Movil,Cambio,Cambio_Brusco
20,O2,22101YV009#S0020,Periodo 1,Oxígeno,21.0,%vol,NaN,NaN,2024-09-06 12:53:44,2024-09-06 12:54:44,NaN,NaN,NaN,NaN,29,NaN,NaN,False
1,NO2,22083AF012#S0004,Periodo 1,Dióxido de nitrógeno,0.0,ppm,NaN,NaN,2024-09-06 12:53:44,2024-09-06 12:54:34,NaN,NaN,NaN,NaN,29,NaN,21.0,False
17,LEL(CH4),22101J7044#S0021,Periodo 1,Gas combustible,0.0,%vol,NaN,NaN,2024-09-06 12:53:44,2024-09-06 12:54:44,NaN,NaN,Metano,NaN,29,NaN,0.0,False
14,H2S,2209366013#S0002,Periodo 1,Sulfuro de hidrógeno,0.0,ppm,NaN,NaN,2024-09-06 12:53:44,2024-09-06 12:54:44,NaN,NaN,NaN,NaN,29,NaN,0.0,False
11,CO,2209366013#S0001,Periodo 1,Monóxido de carbono,0.0,ppm,NaN,NaN,2024-09-06 12:53:44,2024-09-06 12:54:44,NaN,NaN,NaN,NaN,29,NaN,0.0,False
